# NBA Research
---
The goal is to first engineer some predictive features, then filter features out based on importance and redundancy.

Namely, we want to mimic the following features:
```
diff_massey_SAG
diff_massey_POM
diff_massey_WLK
diff_seed_num
diff_massey_DOL
diff_massey_PMW
diff_consensus_rank
diff_massey_MOR
diff_massey_RPI
diff_massey_COL
diff_kg_scoring_margin
diff_kg_wins
diff_kg_margin_last10_delta
diff_kg_win_pct
diff_massey_BPI
diff_kg_losses
diff_path_best_opp_seed
diff_kg_net_strong_margin
diff_path_games_played
```
After engineering them, we will run them through our pipeline.

In [ ]:
import pandas as pd
import requests
import time
from nba_api.stats.endpoints import leaguegamefinder

# --- 1. DISCOVER NBA STATS CODES ---
def get_nba_historical_codes():
    print("🚀 Fetching all historical NBA team codes...")
    # This captures every team tricode appearing in a game record since 1946
    finder = leaguegamefinder.LeagueGameFinder(league_id_nullable='00')
    df_nba = finder.get_data_frames()[0]
    
    # 'TEAM_ABBREVIATION' is the standard tricode in V2/V3
    nba_codes = sorted(df_nba['TEAM_ABBREVIATION'].unique().tolist())
    return nba_codes

# --- 2. DISCOVER ESPN CODES ---
def get_espn_historical_codes(start_year=1947, end_year=2026):
    """
    Samples dates across history to find all unique ESPN abbreviations.
    We sample November 15th of every year to catch relocated/rebranded teams.
    """
    espn_codes = set()
    print(f"🚀 Sampling ESPN codes from {start_year} to {end_year}...")
    
    for year in range(start_year, end_year + 1):
        # Sampling mid-November usually ensures the season is active
        date_str = f"{year}1115"
        url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard?dates={date_str}"
        
        try:
            resp = requests.get(url, timeout=10)
            data = resp.json()
            for event in data.get('events', []):
                for comp in event.get('competitions', [{}])[0].get('competitors', []):
                    code = comp['team']['abbreviation'].upper()
                    espn_codes.add(code)
            print(f"✅ Sampled {year}")
        except Exception as e:
            print(f"❌ Error in {year}: {e}")
        
        time.sleep(0.5)
        
    return sorted(list(espn_codes))

# --- EXECUTION ---
nba_list = get_nba_historical_codes()
espn_list = get_espn_historical_codes()

# Create a comparison table
all_codes = sorted(list(set(nba_list) | set(espn_list)))
comparison = []
for code in all_codes:
    comparison.append({
        "Code": code,
        "In_NBA_API": code in nba_list,
        "In_ESPN_API": code in espn_list
    })

df_compare = pd.DataFrame(comparison)
print("\n--- Team Code Cross-Reference ---")
print(df_compare)

# Save for your mapping logic
df_compare.to_csv('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/team_code_audit.csv', index=False)

In [1]:
import pandas as pd
from nba_api.stats.endpoints import playercareerstats
from nba_api.live.nba.endpoints import scoreboard

# 1. Fetch Nikola Jokić's Career Stats (Stats Endpoint)
# ID 203999 = Nikola Jokić
career = playercareerstats.PlayerCareerStats(player_id='203999')

# Use the 'season_totals_regular_season' dataset specifically
df_jokic = career.season_totals_regular_season.get_data_frame()

# 2. Fetch Today's Scoreboard (Live Endpoint)
# Note: Live endpoints return dictionaries; we convert them to DataFrames manually
board = scoreboard.ScoreBoard()
games_dict = board.get_dict()
games_list = games_dict['scoreboard']['games']

df_scoreboard = pd.DataFrame(games_list)

# 3. Save to CSV
print(f'player cols:\n{df_jokic.columns}')
print(f'scoreboard cols:\n{df_scoreboard.columns}')

player cols:
Index(['PLAYER_ID', 'SEASON_ID', 'LEAGUE_ID', 'TEAM_ID', 'TEAM_ABBREVIATION',
       'PLAYER_AGE', 'GP', 'GS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A',
       'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL',
       'BLK', 'TOV', 'PF', 'PTS'],
      dtype='str')
scoreboard cols:
Index(['gameId', 'gameCode', 'gameStatus', 'gameStatusText', 'period',
       'gameClock', 'gameTimeUTC', 'gameEt', 'regulationPeriods',
       'ifNecessary', 'seriesGameNumber', 'gameLabel', 'gameSubLabel',
       'seriesText', 'seriesConference', 'poRoundDesc', 'gameSubtype',
       'isNeutral', 'homeTeam', 'awayTeam', 'gameLeaders', 'pbOdds'],
      dtype='str')


In [4]:
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams

# 1. Get the Team ID (e.g., for the Lakers)
nba_teams = teams.get_teams()
lakers = [team for team in nba_teams if team['abbreviation'] == 'LAL'][0]
lakers_id = lakers['id']

# 2. Fetch all games for that team for the 2025-26 season
gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=lakers_id,
    season_nullable='2025-26'
)

# 3. Convert to a Pandas DataFrame
games_df = gamefinder.get_data_frames()[0]

# Display recent results
print(f'Columns: {games_df.columns}')

Columns: Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS'],
      dtype='str')


In [39]:
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams

class Timer:
    def __enter__(self): self.start = time.time(); return self
    def __exit__(self, *args): print(f"Time taken: {time.time() - self.start:.4f}s")

# 1. Profile Team Lookup (Local/Static)
with Timer() as t:
    print("Looking up Team ID...")
    nba_teams = teams.get_teams()
    lakers = [team for team in nba_teams if team['abbreviation'] == 'LAL'][0]
    lakers_id = lakers['id']

# 2. Profile API Object Creation (This is usually the bottleneck)
# Even if data is cached or fast, the object instantiation is heavy.
with Timer() as t:
    print("Instantiating LeagueGameFinder Object...")
    gamefinder = leaguegamefinder.LeagueGameFinder(
        team_id_nullable=lakers_id,
        season_nullable='2025-26'
    )

# 3. Profile DataFrame Transformation
with Timer() as t:
    print("Transforming to DataFrame...")
    games_df = gamefinder.get_data_frames()[0]

# 4. Data Inspection (Checking structure and types)
print("\n--- Object Structure ---")
print(f"Dataframe Shape: {games_df.shape}")
print(f"Memory Usage: {games_df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print("\n--- Column Types ---")
print(games_df.dtypes)


Looking up Team ID...
Time taken: 0.0007s
Instantiating LeagueGameFinder Object...
Time taken: 0.2108s
Transforming to DataFrame...
Time taken: 0.0132s

--- Object Structure ---
Dataframe Shape: (94, 28)
Memory Usage: 57.41 KB

--- Column Types ---
SEASON_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                  str
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int

In [38]:
games_df['SEASON_ID'].unique()

<StringArray>
['42025', '22025', '12025']
Length: 3, dtype: str

In [10]:
import os
import pandas as pd
import time
import random
from nba_api.stats.endpoints import leaguegamefinder, boxscoretraditionalv3
from nba_api.stats.library.http import NBAStatsHTTP

# 1. Global Setup
def patch_nba_api_headers():
    NBAStatsHTTP.default_headers = {
        'Host': 'stats.nba.com',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0',
        'Accept': 'application/json, text/plain, */*',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://www.nba.com/',
        'Origin': 'https://www.nba.com',
        'Connection': 'keep-alive',
    }

patch_nba_api_headers()
DATA_DIR = "./nba_data_rosters"
os.makedirs(DATA_DIR, exist_ok=True)

# 2. Get the Master List of Game IDs
print("Fetching master game list...")
finder = leaguegamefinder.LeagueGameFinder(league_id_nullable='00')
all_games = finder.get_data_frames()[0]
unique_game_ids = all_games['GAME_ID'].unique()

# 3. The Scraping Loop with Checkpointing
def run_scraper(game_ids):
    for g_id in game_ids:
        file_path = os.path.join(DATA_DIR, f"{g_id}.csv")
        
        # SKIP if we already have this game (The Resume Logic)
        if os.path.exists(file_path):
            continue
            
        try:
            print(f"Processing Game: {g_id}")
            # NBA servers are fickle; use a long timeout
            box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=g_id, timeout=60)
            df = box.get_data_frames()[0]
            
            if not df.empty:
                df.to_csv(file_path, index=False)
            
            # The Golden Rule: 0.6s to 1.2s jittered sleep
            time.sleep(random.uniform(0.7, 1.3))
            
        except Exception as e:
            print(f"Failed on {g_id}: {e}. Sleeping 10s and continuing...")
            time.sleep(10) # Heavy sleep on error to let the IP cool down
            continue

# Run the scraper
# To test, you can slice the list: unique_game_ids[:100]
run_scraper(unique_game_ids)

Fetching master game list...
Processing Game: 0042500107
Processing Game: 0042500137
Processing Game: 0042500117
Processing Game: 0042500106
Processing Game: 0042500176
Processing Game: 0042500136
Processing Game: 0042500116
Processing Game: 0042500126
Processing Game: 0042500166
Processing Game: 0042500175
Processing Game: 0042500135
Processing Game: 0042500105
Processing Game: 0042500125
Processing Game: 0042500155
Processing Game: 0042500115
Processing Game: 0042500165
Processing Game: 0042500144
Processing Game: 0042500104
Processing Game: 0042500134
Processing Game: 0042500174
Processing Game: 0042500114
Processing Game: 0042500154
Processing Game: 0042500164
Processing Game: 0042500143
Processing Game: 0042500103
Processing Game: 0042500124
Processing Game: 0042500153
Processing Game: 0042500113
Processing Game: 0042500173
Processing Game: 0042500123
Processing Game: 0042500163
Processing Game: 0042500133
Processing Game: 0042500102
Processing Game: 0042500142
Processing Game: 00

KeyboardInterrupt: 

In [11]:
import pandas as pd
from nba_api.stats import endpoints
import time

# 1. Define the endpoints you want to sample
# You can add any from your list here
sample_list = [
    'LeagueDashPlayerClutch', 
    'PlayerEstimatedMetrics', 
    'WinProbabilityPBP',
    'TeamDashboardByGeneralSplits'
]

# 2. Setup Headers (Crucial for 2026 stability)
headers = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0',
    'Referer': 'https://www.nba.com/'
}

def sample_endpoints(endpoint_names):
    for name in endpoint_names:
        print(f"\n--- Sampling: {name} ---")
        try:
            # Dynamically get the class from the endpoints module
            endpoint_class = getattr(endpoints, name)
            
            # Most endpoints require at least one parameter (like LeagueID or Season)
            # We use common defaults here
            instance = endpoint_class(headers=headers, timeout=30)
            
            # Get the first dataframe available in the object
            df = instance.get_data_frames()[0]
            
            print(f"Shape: {df.shape}")
            print(f"Columns: {list(df.columns[:10])}...") # Print first 10 columns
            print(df.head(3))
            
            # Rate limiting safety
            time.sleep(1.5) 
            
        except Exception as e:
            print(f"Could not sample {name}: {e}")

if __name__ == "__main__":
    sample_endpoints(sample_list)


--- Sampling: LeagueDashPlayerClutch ---
Could not sample LeagueDashPlayerClutch: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

--- Sampling: PlayerEstimatedMetrics ---
Could not sample PlayerEstimatedMetrics: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

--- Sampling: WinProbabilityPBP ---
Could not sample WinProbabilityPBP: WinProbabilityPBP.__init__() missing 1 required positional argument: 'game_id'

--- Sampling: TeamDashboardByGeneralSplits ---
Could not sample TeamDashboardByGeneralSplits: TeamDashboardByGeneralSplits.__init__() missing 1 required positional argument: 'team_id'


In [ ]:
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Configuration
file_path = pathlib.Path("data_curation/data/games.parquet")

if not file_path.exists():
    display(Markdown(f"### ❌ File not found\nPath: `{file_path}`"))
else:
    # 1. Load Data
    df = pd.read_parquet(file_path)
    
    # 2. High-Level Summary
    display(Markdown("## 📊 Dataset Overview"))
    stats = pd.DataFrame({
        "Metric": ["Total Games", "Total Columns", "File Size", "Memory Usage"],
        "Value": [
            f"{len(df):,}",
            f"{len(df.columns)}",
            f"{file_path.stat().st_size / (1024*1024):.2f} MB",
            f"{df.memory_usage(deep=True).sum() / (1024*1024):.2f} MB"
        ]
    })
    display(stats)

    # 3. Column Inspection
    display(Markdown("## 🛠 Column Types & Quality"))
    quality = pd.DataFrame({
        "Dtype": df.dtypes,
        "Nulls": df.isnull().sum(),
        "Null %": (df.isnull().sum() / len(df) * 100).round(2)
    })
    display(quality)

    # 4. Temporal Distribution
    if "game_date" in df.columns:
        df["game_date"] = pd.to_datetime(df["game_date"])
        
        display(Markdown(f"## 📅 Time Coverage"))
        display(Markdown(f"**Range:** {df['game_date'].min().date()} to {df['game_date'].max().date()}"))
        
        # Plotting game frequency by month/year
        plt.figure(figsize=(12, 4))
        df.set_index("game_date").resample("ME").size().plot(
            title="Volume of Games Over Time",
            color="#2ecc71",
            lw=2
        )
        plt.ylabel("Game Count")
        plt.grid(axis='y', alpha=0.3)
        plt.show()

    # 5. Data Sample
    display(Markdown("## 🔍 Latest 5 Games"))
    display(df.sort_values("game_date", ascending=False).head())

    # 6. Team Participation
    if "home_team" in df.columns and "away_team" in df.columns:
        display(Markdown("## 🏀 Top 10 Most Frequent Teams"))
        all_teams = pd.concat([df["home_team"], df["away_team"]])
        display(all_teams.value_counts().head(10).to_frame(name="Total Games"))

In [145]:
import pandas as pd
import requests
import time
from datetime import datetime

def compare_hornets_pelicans_history(start_year=2000):
    url = "https://site.api.espn.com/apis/v2/sports/basketball/nba/standings"
    target_ids = ["3", "30"]
    history = []
    
    current_year = datetime.now().year
    print(f"Comparing Franchise IDs {target_ids} from {start_year} to {current_year}...")

    for y in range(start_year, current_year + 1):
        try:
            # Correct URL formatting - params are passed as a dictionary
            r = requests.get(url, params={"season": y}, timeout=10)
            if r.status_code == 200:
                data = r.json()
                for group in data.get("children", []):
                    for entry in group.get("standings", {}).get("entries", []):
                        t = entry.get("team", {})
                        t_id = str(t.get("id"))
                        
                        if t_id in target_ids:
                            history.append({
                                "year": y,
                                "ESPN_ID": t_id,
                                "Name": t.get("displayName"),
                                "Abbr": t.get("abbreviation")
                            })
            time.sleep(0.2) # Polite delay
        except Exception as e:
            print(f"Error for year {y}: {e}")

    df = pd.DataFrame(history)
    # Pivot to see them side-by-side
    comparison = df.pivot(index='year', columns='ESPN_ID', values=['Name', 'Abbr'])
    return comparison

if __name__ == "__main__":
    history_comp = compare_hornets_pelicans_history(1985)
    print(history_comp)


Comparing Franchise IDs ['3', '30'] from 1985 to 2026...
                                 Name                    Abbr     
ESPN_ID                             3                 30    3   30
year                                                              
1989                Charlotte Hornets                NaN  CHA  NaN
1990                Charlotte Hornets                NaN  CHA  NaN
1991                Charlotte Hornets                NaN  CHA  NaN
1992                Charlotte Hornets                NaN  CHA  NaN
1993                Charlotte Hornets                NaN  CHA  NaN
1994                Charlotte Hornets                NaN  CHA  NaN
1995                Charlotte Hornets                NaN  CHA  NaN
1996                Charlotte Hornets                NaN  CHA  NaN
1997                Charlotte Hornets                NaN  CHA  NaN
1998                Charlotte Hornets                NaN  CHA  NaN
1999                Charlotte Hornets                NaN  CHA  NaN
2000 

In [143]:
import pandas as pd

espn = pd.read_csv('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/espn_team_candidates.csv')

# 1. Identify the core ESPN Hornets row (ID '30')
# Using .iloc[0] to get a single row as a template
# hornets_template = espn[espn['ESPN_NAME'].str.contains('Hornets', na=False)].iloc[[0]].copy()

# # 2. Update the row with Bobcats historical metadata
# # We keep the ESPN_ID the same (30) but change the branding
# hornets_template['ESPN_NAME'] = 'Charlotte Bobcats'
# hornets_template['ESPN_ABBR'] = 'CHA'
# hornets_template['year'] = 2004  # Marking the start of the Bobcats era

# # 3. Append back to the original ESPN dataframe
# espn = pd.concat([espn, hornets_template], ignore_index=True)

# # 4. Verification: Check the count of Charlotte-related rows
# print("--- Verification ---")
# print(espn[espn['ESPN_ID'].astype(str) == '30'][['ESPN_NAME', 'year']])
espn = espn.sort_values('ESPN_ID')
espn = espn.dropna()
espn


,ESPN_ID,ESPN_ABBR,ESPN_NAME,year
28,1,ATL,Atlanta Hawks,2026
29,2,BOS,Boston Celtics,2026
54,3,NO,New Orleans Pelicans,2026
24,3,CHA,Charlotte Hornets,2002
40,4,CHI,Chicago Bulls,2026
33,5,CLE,Cleveland Cavaliers,2026
55,6,DAL,Dallas Mavericks,2026
48,7,DEN,Denver Nuggets,2026
27,8,DET,Detroit Pistons,2026
49,9,GS,Golden State Warriors,2026


In [154]:
import pandas as pd

# 1. Load your data
espn = pd.read_csv('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/espn_team_candidates.csv')
nba = pd.read_parquet('../data_curation/data/nba_teams.parquet')

# The Static ID Crosswalk (The only data we can trust)
nba_to_key = {
    "1610612737": "Atlanta", "1610612738": "Boston", "1610612740": "New Orleans",
    "1610612741": "Chicago", "1610612739": "Cleveland", "1610612742": "Dallas",
    "1610612743": "Denver", "1610612765": "Detroit", "1610612744": "Golden State",
    "1610612745": "Houston", "1610612754": "Indiana", "1610612746": "LA Clippers",
    "1610612747": "LA Lakers", "1610612748": "Miami", "1610612749": "Milwaukee",
    "1610612750": "Minnesota", "1610612751": "Brooklyn", "1610612752": "New York",
    "1610612753": "Orlando", "1610612755": "Philadelphia", "1610612756": "Phoenix",
    "1610612757": "Portland", "1610612758": "Sacramento", "1610612759": "San Antonio",
    "1610612760": "Oklahoma City", "1610612762": "Utah", "1610612764": "Washington",
    "1610612761": "Toronto", "1610612763": "Memphis", "1610612766": "Charlotte"
}

espn_to_key = {
    "1": "Atlanta", "2": "Boston", "3": "New Orleans", "4": "Chicago", 
    "5": "Cleveland", "6": "Dallas", "7": "Denver", "8": "Detroit", 
    "9": "Golden State", "10": "Houston", "11": "Indiana", "12": "LA Clippers", 
    "13": "LA Lakers", "14": "Miami", "15": "Milwaukee", "16": "Minnesota", 
    "17": "Brooklyn", "18": "New York", "19": "Orlando", "20": "Philadelphia", 
    "21": "Phoenix", "22": "Portland", "23": "Sacramento", "24": "San Antonio", 
    "25": "Oklahoma City", "26": "Utah", "27": "Washington", "28": "Toronto", 
    "29": "Memphis", "30": "Charlotte"
}

# 1. Clean IDs to string and assign the Franchise Key
nba['FRANCHISE'] = nba['TEAM_ID'].astype(int).astype(str).map(nba_to_key)
espn['FRANCHISE'] = espn['ESPN_ID'].astype(int).astype(str).map(espn_to_key)
print(nba['FRANCHISE'])

# 2. Join on the FRANCHISE key only (ignoring the corrupted year)
# Using 'outer' first to see where we have misses
merged = pd.merge(
    nba[['TEAM_ID', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'FRANCHISE']], 
    espn[['ESPN_ID', 'ESPN_NAME', 'ESPN_ABBR', 'FRANCHISE']], 
    on='FRANCHISE', 
    how='inner'
).drop_duplicates()

# 3. Clean up the final mapping result
final_mapping = merged.sort_values('TEAM_ID')
final_mapping


0           Atlanta
2            Boston
4         Cleveland
6       New Orleans
7       New Orleans
8           Chicago
10           Dallas
12           Denver
15     Golden State
16          Houston
18      LA Clippers
19      LA Clippers
20        LA Lakers
22            Miami
24        Milwaukee
26        Minnesota
29         Brooklyn
28         Brooklyn
31         New York
32          Orlando
34          Indiana
36     Philadelphia
38          Phoenix
40         Portland
42       Sacramento
44      San Antonio
47    Oklahoma City
46    Oklahoma City
48          Toronto
50             Utah
52          Memphis
54       Washington
60       Washington
56          Detroit
58        Charlotte
59        Charlotte
Name: FRANCHISE, dtype: str


,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,FRANCHISE,ESPN_ID,ESPN_NAME,ESPN_ABBR
0,1610612737,Atlanta Hawks,ATL,Atlanta,1,Atlanta Hawks,NaN
1,1610612737,Atlanta Hawks,ATL,Atlanta,1,Atlanta Hawks,ATL
2,1610612738,Boston Celtics,BOS,Boston,2,Boston Celtics,BOS
3,1610612739,Cleveland Cavaliers,CLE,Cleveland,5,Cleveland Cavaliers,CLE
4,1610612740,New Orleans Hornets,NOH,New Orleans,3,Charlotte Hornets,CHA
5,1610612740,New Orleans Hornets,NOH,New Orleans,3,New Orleans Pelicans,NO
6,1610612740,New Orleans Pelicans,NOP,New Orleans,3,Charlotte Hornets,CHA
7,1610612740,New Orleans Pelicans,NOP,New Orleans,3,New Orleans Pelicans,NO
8,1610612741,Chicago Bulls,CHI,Chicago,4,Chicago Bulls,CHI
9,1610612742,Dallas Mavericks,DAL,Dallas,6,Dallas Mavericks,DAL


In [155]:
final_mapping.to_parquet('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/team_mappings.parquet')

In [ ]:
import pandas as pd
from nba_api.stats.static import teams

# Fetch the list of dictionaries and convert to a DataFrame
all_teams_df = pd.DataFrame(teams.get_teams())

# Print the entire table
team_ids = all_teams_df['id']


df = pd.read_csv('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/nba_team_candidates.csv')

# Filter to keep ALL rows that share a non-unique ID
merged = pd.merge(df, team_ids, left_on='TEAM_ID', right_on='id', how='right')

# 1. Isolate the Bullets rows into a new temporary DataFrame
bullets_rows = merged[merged['TEAM_NAME'] == 'Washington Wizards'].copy()

# 2. Rename the team in the temporary DataFrame
bullets_rows['TEAM_NAME'] = 'Washington Bullets'

# 3. Append (concatenate) the new rows back to the original DataFrame
merged = pd.concat([merged, bullets_rows], ignore_index=True)

merged = merged.sort_values(by='id')
merged = merged.drop(columns=['id', 'first_seen_season'])

merged = merged.drop_duplicates()

print(f"Unique IDs: {len(merged['TEAM_ID'].unique())}")

# merged.to_parquet('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/nba_teams.parquet')

merged

Unique IDs: 30


,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME
0,1610612737,ATL,Atlanta Hawks
2,1610612738,BOS,Boston Celtics
4,1610612739,CLE,Cleveland Cavaliers
6,1610612740,NOH,New Orleans Hornets
7,1610612740,NOP,New Orleans Pelicans
8,1610612741,CHI,Chicago Bulls
10,1610612742,DAL,Dallas Mavericks
12,1610612743,DEN,Denver Nuggets
15,1610612744,GSW,Golden State Warriors
16,1610612745,HOU,Houston Rockets


In [171]:
import pandas as pd
pd.set_option('display.min_rows', 10)
pd.set_option('display.max_rows', 10)

games = pd.read_parquet('../data_curation/data/game_map.parquet')

keys = games[~games['espn_map_keys'].isna()]

keys

,game_id,espn_event_id,game_date_et,home_abbr_nba,away_abbr_nba,home_abbr_espn,away_abbr_espn,match_method,espn_map_keys
53246,0012500008,NaN,2025-10-02,NYK,NYK,NY,NY,no_match,"[('NY', 'PHI'), ('PHI', 'NY'), ('NO', 'MEL'), ..."
53251,0012500011,NaN,2025-10-04,NOP,SEM,NO,NaN,no_match,"[('PHI', 'NY'), ('NY', 'PHI'), ('BKN', 'HAPOEL..."
53252,0012500010,NaN,2025-10-04,PHI,PHI,PHI,PHI,no_match,"[('PHI', 'NY'), ('NY', 'PHI'), ('BKN', 'HAPOEL..."
53253,0012500026,NaN,2025-10-04,BKN,HAP,BKN,NaN,no_match,"[('PHI', 'NY'), ('NY', 'PHI'), ('BKN', 'HAPOEL..."
53261,0012500032,NaN,2025-10-06,SAS,GUA,SA,NaN,no_match,"[('MIA', 'MIL'), ('MIL', 'MIA'), ('HOU', 'ATL'..."
...,...,...,...,...,...,...,...,...,...
54140,0032500005,NaN,2026-02-13,TMC,VIN,NaN,NaN,no_match,[]
54141,0032500041,NaN,2026-02-15,STP,STR,NaN,NaN,no_match,"[('STARS', 'WORLD'), ('WORLD', 'STARS'), ('STR..."
54142,0032500011,NaN,2026-02-15,STR,WLD,NaN,NaN,no_match,"[('STARS', 'WORLD'), ('WORLD', 'STARS'), ('STR..."
54143,0032500031,NaN,2026-02-15,STP,WLD,NaN,NaN,no_match,"[('STARS', 'WORLD'), ('WORLD', 'STARS'), ('STR..."


In [ ]:
# analyze games parquet
import pandas as pd

games = pd.read_parquet('/Users/michaelharoon/Projects/Prediction markets/nba/data_curation/data/games.parquet')
games = games[~games['game_code'].str.contains('2026')]
games

Successfully saved 278 columns to games_columns.txt
